In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, count, col
import utils.bronze_preprocess as bp

In [2]:
def create_session():
    spk = SparkSession.builder \
        .appName('EDA') \
        .getOrCreate()
    return spk

In [3]:
spark = create_session()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 08:09:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.parquet('datamart/bronze/lms_loan_daily/lms_loan_daily_*')
df.show(5)

26/05/19 08:09:56 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: datamart/bronze/lms_loan_daily/lms_loan_daily_*.
java.io.FileNotFoundException: File datamart/bronze/lms_loan_daily/lms_loan_daily_* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala

+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|             loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|CUS_0x500d_2024_1...| CUS_0x500d|     2024-12-01|    10|              1|   10000| 1000.0|     0.0|     1000.0|10000.0|   2025-01-01|
|CUS_0x501c_2024_0...| CUS_0x501c|     2024-03-01|    10|             10|   10000| 1000.0|     0.0|     7000.0| 7000.0|   2025-01-01|
|CUS_0x503f_2025_0...| CUS_0x503f|     2025-01-01|    10|              0|   10000|    0.0|     0.0|        0.0|10000.0|   2025-01-01|
|CUS_0x5048_2024_0...| CUS_0x5048|     2024-03-01|    10|             10|   10000| 1000.0|     0.0|     7000.0| 7000.0|   2025-01-01|
|CUS_0x504f_2024_1...| CUS_0x504f|     2024-11-01|    10|     

In [5]:
df.describe()

DataFrame[summary: string, loan_id: string, Customer_ID: string, tenure: string, installment_num: string, loan_amt: string, due_amt: string, paid_amt: string, overdue_amt: string, balance: string]

In [6]:
df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: integer (nullable = true)
 |-- due_amt: double (nullable = true)
 |-- paid_amt: double (nullable = true)
 |-- overdue_amt: double (nullable = true)
 |-- balance: double (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [7]:
df.select([count(when(col(i).isNull(), i)).alias(i) for i in df.columns]).show()

+-------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+-------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|      0|          0|              0|     0|              0|       0|      0|       0|          0|      0|            0|
+-------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+



In [8]:
df.summary().show()

26/05/19 08:10:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:==============>                                            (2 + 6) / 8]

+-------+--------------------+-----------+------+------------------+--------+-----------------+-----------------+------------------+------------------+
|summary|             loan_id|Customer_ID|tenure|   installment_num|loan_amt|          due_amt|         paid_amt|       overdue_amt|           balance|
+-------+--------------------+-----------+------+------------------+--------+-----------------+-----------------+------------------+------------------+
|  count|              137500|     137500|137500|            137500|  137500|           137500|           137500|            137500|            137500|
|   mean|                NULL|       NULL|  10.0|               5.0| 10000.0|909.0909090909091| 711.810909090909| 871.9054545454545| 5871.905454545455|
| stddev|                NULL|       NULL|   0.0|3.1622891594225964|     0.0|287.4808326747818|485.7268593783832|2002.6725443189514|3070.7775745457793|
|    min|CUS_0x1000_2023_0...| CUS_0x1000|    10|                 0|   10000|           

In [9]:
df.groupby(['loan_id', 'installment_num', 'snapshot_date']).count().where('count > 1').sort('count', ascending=True).show()

+-------+---------------+-------------+-----+
|loan_id|installment_num|snapshot_date|count|
+-------+---------------+-------------+-----+
+-------+---------------+-------------+-----+



In [10]:
fc = spark.read.parquet('datamart/bronze/feature_clickstream/feature_clickstream_*')
fc.show(5)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
| 161| 165| 100| 146| 211|  37| 119| 131| 170|  202|  104|  281|   63|   79|   70|  101|  210|  -90|   28|  172| CUS_0x1037|   2024-09-01|
| 129| -53|  95|  72| 177| 127| 185| 205|  84|  134|  127|  304|   26|  128|  129|  236|  164|   39|  234|  358| CUS_0x1069|   2024-09-01|
|  46| 121| 148|  61| 109| 127|  23|  41|   3|   68|   99|  177|   54|  -95|  166|  137|  332|   29|   27|   50| CUS_0x114a|   2024-09-01|
|  81|  17| -41|  15| -50| -20| 104| 110|  13|  259|  124|   64|  250|  101|  131|  121|  173|   75|  101| -175| CUS_0x1184|   2024-09-01|
| 367| 103| 177| 227|  82| 

26/05/19 08:10:04 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: datamart/bronze/feature_clickstream/feature_clickstream_*.
java.io.FileNotFoundException: File datamart/bronze/feature_clickstream/feature_clickstream_* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Res

In [14]:
fc.describe().toPandas().T

,0,1,2,3,4
summary,count,mean,stddev,min,max
fe_1,215376,101.41479552039225,99.83359392844191,-378,541
fe_2,215376,103.09619456206819,99.93000203749274,-356,560
fe_3,215376,104.33370941980536,100.59986549187684,-399,583
fe_4,215376,105.64850308298047,100.3260647704614,-307,562
fe_5,215376,106.99667558130896,100.69360665858567,-343,570
fe_6,215376,103.23592229403462,100.27038751891186,-321,565
fe_7,215376,107.0703374563554,100.32326515826551,-368,537
fe_8,215376,110.71872446326425,100.24369781781941,-361,573
fe_9,215376,114.40635446846446,100.18613852691682,-328,577


In [12]:
fc.printSchema()

root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [13]:
fc.select([count(when(col(i).isNull(), i)).alias(i) for i in fc.columns]).show()

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0|    0|    0|    0|    0|    0|    0|    0|    0|          0|            0|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+



In [16]:
fc.summary().toPandas().T

,0,1,2,3,4,5,6,7
summary,count,mean,stddev,min,25%,50%,75%,max
fe_1,215376,101.41479552039225,99.83359392844191,-378,34,102,169,541
fe_2,215376,103.09619456206819,99.93000203749274,-356,36,103,170,560
fe_3,215376,104.33370941980536,100.59986549187684,-399,36,104,172,583
fe_4,215376,105.64850308298047,100.3260647704614,-307,38,106,173,562
fe_5,215376,106.99667558130896,100.69360665858567,-343,39,107,175,570
fe_6,215376,103.23592229403462,100.27038751891186,-321,36,103,171,565
fe_7,215376,107.0703374563554,100.32326515826551,-368,39,107,174,537
fe_8,215376,110.71872446326425,100.24369781781941,-361,43,111,179,573
fe_9,215376,114.40635446846446,100.18613852691682,-328,47,115,182,577


In [17]:
fc.groupby(['Customer_ID', 'snapshot_date']).count().where('count > 1').show()

+-----------+-------------+-----+
|Customer_ID|snapshot_date|count|
+-----------+-------------+-----+
+-----------+-------------+-----+



In [19]:
fa = spark.read.parquet('datamart/bronze/features_attributes/features_attributes_*')
fa.show(5)

26/05/19 08:31:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: datamart/bronze/features_attributes/features_attributes_*.
java.io.FileNotFoundException: File datamart/bronze/features_attributes/features_attributes_* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Res

+-----------+-----------+---+-----------+------------+-------------+
|Customer_ID|       Name|Age|        SSN|  Occupation|snapshot_date|
+-----------+-----------+---+-----------+------------+-------------+
| CUS_0x10ac|      Zhouy| 29|780-50-4730|   Developer|   2024-08-01|
| CUS_0x10c5|      Moony| 24|041-74-6785|     _______|   2024-08-01|
| CUS_0x1145|Blenkinsopr| 24|426-31-9194|     Teacher|   2024-08-01|
| CUS_0x11ac|  Liana B.v|26_|835-92-7751|  Journalist|   2024-08-01|
| CUS_0x122c| Papadimasf| 48|883-73-9594|Entrepreneur|   2024-08-01|
+-----------+-----------+---+-----------+------------+-------------+
only showing top 5 rows


In [34]:
fa.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [20]:
fa.summary().show()

+-------+-----------+------+------------------+-----------+----------+
|summary|Customer_ID|  Name|               Age|        SSN|Occupation|
+-------+-----------+------+------------------+-----------+----------+
|  count|      12500| 12500|             12500|      12500|     12500|
|   mean|       NULL|  NULL|104.40925566888646|       NULL|      NULL|
| stddev|       NULL|  NULL| 659.4794049008864|       NULL|      NULL|
|    min| CUS_0x1000| Mattr|              -500|  #F%$D@*&8|Accountant|
|    25%|       NULL|  NULL|              25.0|       NULL|      NULL|
|    50%|       NULL|  NULL|              33.0|       NULL|      NULL|
|    75%|       NULL|  NULL|              42.0|       NULL|      NULL|
|    max|  CUS_0xffd|    yv|               925|999-99-3421|   _______|
+-------+-----------+------+------------------+-----------+----------+



In [22]:
fa.select([count(when(col(i).isNull(), i)).alias(i) for i in fa.columns]).show()

+-----------+----+---+---+----------+-------------+
|Customer_ID|Name|Age|SSN|Occupation|snapshot_date|
+-----------+----+---+---+----------+-------------+
|          0|   0|  0|  0|         0|            0|
+-----------+----+---+---+----------+-------------+



In [23]:
fa.groupby(['Customer_ID', 'SSN', 'snapshot_date']).count().where('count > 1').show()

+-----------+---+-------------+-----+
|Customer_ID|SSN|snapshot_date|count|
+-----------+---+-------------+-----+
+-----------+---+-------------+-----+



In [32]:
ff = spark.read.parquet('datamart/bronze/features_financials/features_financials_*')
ff.toPandas().T

26/05/19 09:17:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: datamart/bronze/features_financials/features_financials_*.
java.io.FileNotFoundException: File datamart/bronze/features_financials/features_financials_* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Res

,0,1,2,3,4,5,6,7,8,9,...,12490,12491,12492,12493,12494,12495,12496,12497,12498,12499
Customer_ID,CUS_0x10ac,CUS_0x10c5,CUS_0x1145,CUS_0x11ac,CUS_0x122c,CUS_0x1274,CUS_0x1288,CUS_0x12cc,CUS_0x1338,CUS_0x1370,...,CUS_0xc723,CUS_0xc734,CUS_0xcc4,CUS_0xd07,CUS_0xd29,CUS_0xd50,CUS_0xde4,CUS_0xe16,CUS_0xe7a,CUS_0xff6
Annual_Income,16718.645,158623.16,17975.32,69136.54,36346.13,49070.94,60938.13,20521.36,20078.47,10216.36,...,33386.25,27305.93,82240.98,30608.63,10486.2,43771.85,75402.52,37976.68,7711.97,117639.92
Monthly_Inhand_Salary,1270.220417,13180.596667,1444.943333,5902.378333,3188.844167,4128.245,5163.1775,1721.113333,1447.205833,587.015817,...,2851.1875,2289.494167,6116.362098,2650.719167,592.85,3600.654167,6221.543333,2997.723333,395.664167,9727.326667
Num_Bank_Accounts,7,2,6,3,8,8,10,10,10,10,...,7,3,3,3,6,4,2,9,10,5
Num_Credit_Card,3,7,5,4,6,4,8,8,10,8,...,5,4,5,6,6,3,4,9,5,6
Interest_Rate,15,10,32,14,11,4041,31,17,31,22,...,11,19,3,9,6,9,3,18,29,1
Num_of_Loan,4,1,9,0,3,3_,8,3,8,5,...,3,0,1,1,3,1,3,7,7,2
Type_of_Loan,"Credit-Builder Loan, Credit-Builder Loan, Home...",Payday Loan,"Student Loan, Payday Loan, Not Specified, Mort...",NaN,"Credit-Builder Loan, Home Equity Loan, and Aut...","Student Loan, Payday Loan, and Payday Loan","Payday Loan, Payday Loan, Not Specified, Stude...","Personal Loan, Payday Loan, and Mortgage Loan","Student Loan, Auto Loan, Home Equity Loan, Cre...","Mortgage Loan, Personal Loan, Student Loan, St...",...,"Home Equity Loan, Mortgage Loan, and Mortgage ...",NaN,Home Equity Loan,Mortgage Loan,"Debt Consolidation Loan, Payday Loan, and Debt...",Student Loan,"Not Specified, Auto Loan, and Mortgage Loan","Auto Loan, Personal Loan, Student Loan, Person...","Debt Consolidation Loan, Home Equity Loan, Per...","Home Equity Loan, and Auto Loan"
Delay_from_due_date,26,10,61,14,21,13,26,21,43,20,...,26,22,9,10,21,-2,0,56,57,-3
Num_of_Delayed_Payment,11,5,25,18,15,17,21,12,22,23,...,10,17,3,14,10,1,5_,24,21,7


In [28]:
ff.summary().toPandas().T

,0,1,2,3,4,5,6,7
summary,count,mean,stddev,min,25%,50%,75%,max
Customer_ID,12500,NaN,NaN,CUS_0x1000,NaN,NaN,NaN,CUS_0xffd
Annual_Income,12500,165302.1380547202,1318317.705695486,10003.495_,19369.96,37367.34,72841.32,99995.22
Monthly_Inhand_Salary,12500,4188.5923027131585,3180.1476109204164,303.6454166666666,1624.7516666666668,3086.7566666666667,5947.32,15204.633333333331
Num_Bank_Accounts,12500,16.93992,114.35081529709495,-1,3,6,7,1756
Num_Credit_Card,12500,23.17272,132.00586642912248,0,4,5,7,1499
Interest_Rate,12500,73.21336,468.6822271005273,1,8,14,20,5789
Num_of_Loan,12500,2.8950071566894,65.49885397088403,-100,1.0,3.0,5.0,9_
Type_of_Loan,11074,NaN,NaN,Auto Loan,NaN,NaN,NaN,"Student Loan, and Student Loan"
Delay_from_due_date,12500,21.06088,14.863091419503666,-5,10,18,28,67


In [31]:
ff.select([count(when(col(i).isNull(), i)).alias(i) for i in ff.columns]).toPandas().T

,0
Customer_ID,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0
Num_of_Loan,0
Type_of_Loan,1426
Delay_from_due_date,0
Num_of_Delayed_Payment,0


In [33]:
ff.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: double (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: double (nullable = true)
 |-- Amount_invested_monthly: string (nullable = true)
 |-- Payment_Behaviour: string (nullable = true)
 |-- Mont

In [35]:
ff.groupby(['Customer_ID', 'snapshot_date']).count().where('count > 1').show()

+-----------+-------------+-----+
|Customer_ID|snapshot_date|count|
+-----------+-------------+-----+
+-----------+-------------+-----+

